# Daily Series Analysis

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import mlflow
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from eda_utils import line_with_slider, plot_decomposition, plot_with_residuals
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing as HoltWinters
from statsmodels.tsa.seasonal import STL, seasonal_decompose

from heliocast.data import drop_incomplete_tail, read_sarah3
from heliocast.metrics import calculate_metrics
from heliocast.storage import get_location_prefix
from heliocast.utils import LOCATIONS

px.defaults.height = 700

/home/mario/workspace/solar-predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Setup mlflow
experiment_name = "daily_series_analysis"

mlflow.set_tracking_uri(f"sqlite:///{Path.cwd().parent / 'mlflow.db'}")
exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    mlflow.create_experiment(
        experiment_name, artifact_location=str(Path.cwd().parent / "mlruns")
    )

mlflow.set_experiment(experiment_name)

# Load data

skopje_lp = get_location_prefix(LOCATIONS["SKOPJE"]["LAT"], LOCATIONS["SKOPJE"]["LONG"])
csvs = sorted((Path.cwd().parent / "data" / skopje_lp).glob("year=*/sarah3.csv"))
raw_df = read_sarah3(None, csvs=csvs, read_fn=pd.read_csv)

2026/11/20 22:01:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/20 22:01:38 INFO mlflow.store.db.utils: Updating database tables


In [4]:
df = drop_incomplete_tail(raw_df)
daily_df = df.resample("D").mean()

daily_df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation
datetime,,,,
2026-11-12 00:00:00+00:00,1.794484e+09,57.125000,8.145833,48.979167
2026-11-13 00:00:00+00:00,1.794570e+09,103.750000,65.041667,38.708333
2026-11-14 00:00:00+00:00,1.794657e+09,28.520833,0.333333,28.187500
2026-11-15 00:00:00+00:00,1.794743e+09,73.895833,17.479167,56.416667
2026-11-16 00:00:00+00:00,1.794787e+09,0.000000,0.000000,0.000000


# Smoothing

## Simple Moving Average (SMA) and Exponential Moving Average (EMA)

In [5]:
windows = (5, 10, 30)

ma_df = daily_df[["shortwave_radiation"]].copy()
ma_df["prev"] = ma_df["shortwave_radiation"].shift(1)
metrics = []

for w in windows:
    ma_df[f"sma_{w}"] = ma_df["prev"].rolling(w).mean()
    ma_df[f"ema_{w}"] = ma_df["prev"].ewm(span=w, adjust=False, min_periods=w).mean()

    for algorithm in ("sma", "ema"):
        params = {"algorithm": algorithm, "window": w}
        column = f"{algorithm}_{w}"

        with mlflow.start_run(run_name=column):
            run_metrics = calculate_metrics(ma_df["shortwave_radiation"], ma_df[column])
            run_fig = plot_with_residuals(
                ma_df["shortwave_radiation"],
                ma_df[column],
                window=pd.DateOffset(years=1),
                title=f"{algorithm.upper()} (window={w})",
            )
            mlflow.log_params(params)
            mlflow.log_metrics(run_metrics)
            run_fig.write_html("chart.html")
            mlflow.log_figure(figure=run_fig, artifact_file="chart.html")

        metrics.append({**params, **run_metrics})


colors = ["lightgray", *px.colors.qualitative.Plotly]
fig = make_subplots(
    rows=len(windows),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
)
for row, w in enumerate(windows, start=1):
    columns = {
        "shortwave_radiation": "shortwave_radiation",
        "sma": f"sma_{w}",
        "ema": f"ema_{w}",
    }
    for (name, column), color in zip(columns.items(), colors):
        fig.add_trace(
            go.Scatter(
                x=ma_df.index,
                y=ma_df[column],
                name=name,
                line_color=color,
                legendgroup=name,
                showlegend=row == 1,
            ),
            row=row,
            col=1,
        )
    fig.update_yaxes(title_text=f"{w} days", row=row, col=1)

fig.update_xaxes(range=[pd.to_datetime("2007-01-01"), pd.to_datetime("2008-01-01")])
fig.update_xaxes(
    rangeslider={"visible": True, "thickness": 0.05}, row=len(windows), col=1
)
fig.update_layout(
    title="Shortwave Radiation Moving Averages by Window Size",
    height=800,
    legend={
        "orientation": "h",
        "y": 1.02,
        "x": 1,
        "xanchor": "right",
        "yanchor": "bottom",
    },
)
fig.show()

display(pd.DataFrame(metrics))

,algorithm,window,sse,mse,rmse,residual_std,residual_mean
0,sma,5,4.845569e+07,3609.362639,60.077971,60.080205,-0.022951
1,ema,5,4.373176e+07,3250.223348,57.010730,57.012849,-0.000868
2,sma,10,4.743968e+07,3542.918729,59.522422,59.524643,0.015062
3,ema,10,4.399750e+07,3271.189777,57.194316,57.196442,-0.004592
4,sma,30,5.216707e+07,3937.137477,62.746613,62.748976,-0.025562
5,ema,30,5.026340e+07,3742.621100,61.176965,61.179242,-0.013094


Conclusion: Both methods (SMA and EMA) follow the same trend, and larger windows are smoother but react later.
The averages use only previous days, so they don't leak the predicted value.
Model validation error will show which method and window to use.

## Alpha in EMA

In [6]:
ema_df = daily_df[["shortwave_radiation"]].copy()
prev = ema_df["shortwave_radiation"].shift(1)
metrics = []

for alpha in [0.02, 0.08]:
    params = {"algorithm": "ema", "alpha": alpha}
    column = f"ema_{alpha}"

    with mlflow.start_run(run_name=f"ema_alpha_{alpha}"):
        ema_df[column] = prev.ewm(alpha=alpha, adjust=False).mean()
        run_metrics = calculate_metrics(ema_df["shortwave_radiation"], ema_df[column])
        run_fig = plot_with_residuals(
            ema_df["shortwave_radiation"],
            ema_df[column],
            window=pd.DateOffset(years=1),
            title=f"EMA (alpha={alpha})",
        )
        mlflow.log_params(params)
        mlflow.log_metrics(run_metrics)
        run_fig.write_html("chart.html")
        mlflow.log_figure(figure=run_fig, artifact_file="chart.html")

    metrics.append({**params, **run_metrics})


fig = line_with_slider(
    df=ema_df,
    window=pd.DateOffset(years=1),
    color_discrete_map={"shortwave_radiation": "lightgray"},
)
fig.show()

display(pd.DataFrame(metrics))

,algorithm,alpha,sse,mse,rmse,residual_std,residual_mean
0,ema,0.02,8.586657e+07,6379.862553,79.874042,79.875544,-0.483765
1,ema,0.08,4.796586e+07,3563.850096,59.697991,59.700190,-0.047535


## Holt-Winter

### Prepare data
Remove invalid data points and interpolate missing values.

In [7]:
non_nan_df = daily_df[["shortwave_radiation"]].copy()
# Remove zeroes
non_nan_df.loc[non_nan_df["shortwave_radiation"] == 0] = np.nan

non_nan_df["shortwave_radiation"] = non_nan_df["shortwave_radiation"].interpolate(
    method="time"
)

hw_df = non_nan_df.copy()

### Additive Season

In [8]:
params = {"trend": None, "seasonal_periods": 365, "seasonal": "add"}

with mlflow.start_run(run_name="holtwinters_add"):
    hw = HoltWinters(hw_df["shortwave_radiation"], **params).fit()
    hw_df["holt_winters"] = hw.fittedvalues

    fig = plot_with_residuals(
        hw_df["shortwave_radiation"],
        hw_df["holt_winters"],
        window=pd.DateOffset(years=1),
        title="Holt-Winters (additive)",
    )
    metrics = {
        **calculate_metrics(hw_df["shortwave_radiation"], hw_df["holt_winters"]),
        "aic": hw.aic,
    }

    mlflow.log_params({"algorithm": "holtwinters", **params})
    mlflow.log_metrics(metrics)

    fig.write_html("chart.html")
    mlflow.log_figure(figure=fig, artifact_file="chart.html")

fig.show()
display(pd.DataFrame([metrics]))

,sse,mse,rmse,residual_std,residual_mean,aic
0,4.183521e+07,3106.036789,55.731829,55.733898,-0.005768,109039.614263


### Multiplicative Season

In [9]:
params = {"trend": None, "seasonal_periods": 365, "seasonal": "mul"}

with mlflow.start_run(run_name="holtwinters_mul"):
    hw = HoltWinters(hw_df["shortwave_radiation"], **params).fit()
    hw_df["holt_winters"] = hw.fittedvalues

    fig = plot_with_residuals(
        hw_df["shortwave_radiation"],
        hw_df["holt_winters"],
        window=pd.DateOffset(years=1),
        title="Holt-Winters (multiplicative)",
    )
    metrics = {
        **calculate_metrics(hw_df["shortwave_radiation"], hw_df["holt_winters"]),
        "aic": hw.aic,
    }
    mlflow.log_params({"algorithm": "holtwinters", **params})
    mlflow.log_metrics(metrics)

    fig.write_html("chart.html")
    mlflow.log_figure(figure=fig, artifact_file="chart.html")

fig.show()
display(pd.DataFrame([metrics]))

/home/mario/workspace/solar-predictor/.venv/lib/python3.14/site-packages/statsmodels/tsa/holtwinters/model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)


,sse,mse,rmse,residual_std,residual_mean,aic
0,4.203167e+07,3120.622742,55.862534,55.862808,-0.448474,109102.716653


# Trend, Seasonal and Residual Decomposition

Seasonal period of 365 is chosen because the data is daily and solar radiation repeats every year, so one cycle is 365 days.

## Classical Multiplicative Decomposition

In [10]:
dec_df = non_nan_df.copy()

res = seasonal_decompose(
    dec_df["shortwave_radiation"], model="multiplicative", period=365
)

fig = plot_decomposition(
    res.trend,
    res.seasonal,
    res.resid,
    window=pd.DateOffset(years=10),
    title="Multiplicative decomposition (period=365)",
)
fig.show()

## Seasonal-Trend Decomposition with LOESS

In [11]:
res = STL(dec_df["shortwave_radiation"], period=365).fit()

fig = plot_decomposition(
    res.trend,
    res.seasonal,
    res.resid,
    window=pd.DateOffset(years=10),
    title="STL decomposition (period=365)",
)
fig.show()

Conclusion: The seasonal component captures the yearly cycle of the sun.
The trend is not flat, but it changes only slightly, following sunnier and cloudier years.
The residual is mostly weather: cloudy days and snow in winter cause large drops in radiation.
These weather effects can't be explained by past radiation alone.